In [1]:
import pandas as pd
import numpy as np

In [2]:
subset_dataset = pd.read_parquet("s3://open-jobs-lake/job_quality/welsh_analysis/subset_green_sectors_quality_combined.parquet")

In [3]:
counts_dataset = pd.read_parquet("s3://open-jobs-lake/job_quality/welsh_analysis/country_sectors_counts.parquet")

In [4]:
total_ads_per_country = counts_dataset.groupby('country', dropna=False)['job_id'].sum().to_dict()
total_ads_per_country

{'England': 5458040, 'Scotland': 197598, 'Wales': 139337, nan: 172254}

environmental consultant. these make up 2% of job adverts in England, 3% in Scotland, 4% in Wales. In England the job quality measure of 'career progression' is mentioned in 15% of job adverts, in Wales this is 25%, in Scotland its 10%.... the job quality measure of 'L&D' is mentioned in.... etc
solar installer ...


In [5]:
subset_dataset["country"] = subset_dataset['itl_1_name'].apply(
    lambda x: x if x in ["Wales", "Scotland", None] else "England")
subset_dataset["country"].value_counts(dropna=False)

England     41801
Scotland     2535
NaN          1898
Wales        1317
Name: country, dtype: int64

In [6]:
subset_dataset['GREEN/NOT GREEN'] = subset_dataset['GREEN/NOT GREEN']=='Green'

## Per sector aggregates

In [7]:
sectors_ordered = subset_dataset['sector'].value_counts().index # From most to least frequent

In [19]:
subset_dataset['mid_annualised_salary'] = subset_dataset[['min_annualised_salary', 'max_annualised_salary']].mean(skipna=True,axis =1)

In [20]:
all_sectors_aggs = pd.DataFrame()

for sector_name in sectors_ordered:
    sector_data = subset_dataset[subset_dataset['sector']==sector_name]
    per_sector_aggs = sector_data.groupby("country", dropna=False).agg(
        {
            'job_id': 'count',
            'CAREER': 'sum',
            'COMP': 'sum',
            'FLEX_HOURS': 'sum',
            'FLEX_LOC': 'sum',
            'L&D': 'sum',
            'LEAVE': 'sum',
            'PERKS': 'sum',
            'HOURS': 'sum',
            'PROP_GREEN': 'mean',
            'GREEN TIMESHARE': 'mean',
            'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
            'GREEN/NOT GREEN': 'mean',
            'min_annualised_salary': 'mean',
            'max_annualised_salary': 'mean',
            'mid_annualised_salary': 'mean',
        }).reset_index()
    per_sector_aggs["sector"] = sector_name.replace("&amp;", "&")
    all_sectors_aggs = pd.concat([all_sectors_aggs, per_sector_aggs])

In [21]:
# Get proportions of job adverts
qual_cols = ['CAREER', 'COMP', 'FLEX_HOURS', 'FLEX_LOC', 'L&D',
       'LEAVE', 'PERKS', 'HOURS']
all_sectors_aggs[qual_cols] = all_sectors_aggs[qual_cols].div(all_sectors_aggs['job_id'], axis=0)*100

all_sectors_aggs['total_ads_per_country'] = all_sectors_aggs['country'].map(total_ads_per_country)
all_sectors_aggs['perc_ads_per_country'] = all_sectors_aggs['job_id']*100/all_sectors_aggs['total_ads_per_country']

# Turn proportions to percentages
all_sectors_aggs["PROP_GREEN"] = all_sectors_aggs["PROP_GREEN"]*100
all_sectors_aggs["GREEN/NOT GREEN"] = all_sectors_aggs["GREEN/NOT GREEN"]*100

# Clean up column names
all_sectors_aggs.rename(columns={
    "job_id": "Number of job adverts",
    'PROP_GREEN': "Average percentage of green skills",
    'GREEN TIMESHARE': "Average percentage of time spent on green tasks",
    'INDUSTRY GHG PER UNIT EMISSIONS': "Average GHG emissions",
    'GREEN/NOT GREEN': "Average percentage of jobs which are 'green' (O*NET)",
    'min_annualised_salary': "Average minimum salary",
    'max_annualised_salary': "Average maximum salary",
    'mid_annualised_salary': "Average salary mid point",
}, inplace=True)

In [22]:
all_sectors_aggs[pd.notnull(all_sectors_aggs['country'])].round(3).to_csv("welsh_quality_aggs.csv")

In [23]:
all_sectors_aggs[all_sectors_aggs['country']=="Wales"].round(3).to_csv("welsh_only_quality_aggs.csv")

In [24]:
melted_df = pd.melt(all_sectors_aggs, id_vars=['country', 'sector', 'Number of job adverts'], value_vars=qual_cols).round(3)
melted_df = melted_df[pd.notnull(melted_df['country'])]
melted_df.to_csv("welsh_quality_aggs_melt.csv", index=False)

In [25]:
salary_melted_df = pd.melt(
    all_sectors_aggs.reset_index(drop=True).reset_index(),
    id_vars=['country', 'sector', 'index'], value_vars=["Average minimum salary", "Average maximum salary"]).round()
salary_melted_df = salary_melted_df[pd.notnull(salary_melted_df['country'])]
salary_melted_df.to_csv("welsh_salary_aggs_melt.csv", index=False)

In [26]:
salary_melted_df = pd.melt(
    all_sectors_aggs.reset_index(drop=True).reset_index(),
    id_vars=['country', 'sector', 'index'], value_vars=["Average salary mid point"]).round()
salary_melted_df = salary_melted_df[pd.notnull(salary_melted_df['country'])]
salary_melted_df.to_csv("welsh_mid_salary_aggs_melt.csv", index=False)

In [156]:
totals_melted_df = all_sectors_aggs[['country', 'sector', "Number of job adverts", "perc_ads_per_country"]].round(4)
totals_melted_df = totals_melted_df[pd.notnull(totals_melted_df['country'])]
totals_melted_df.to_csv("welsh_totals_aggs.csv", index=False)